# Individual patient model

Loop the model over the patient prototypes. Generate the averaged mRS distributions for each patient for each stroke unit. Find which of the models is the most representative of the averaged scores.

NOTE - hard-coding no atrial fibrillation diagnosis to match no anticoagulant in the proto patient data.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os

from utils.individual_patient import IndividualPatientModel

## Choose to use artificial or real data

This repository only contains artificial data

In [2]:
use_artificial_data = False

if use_artificial_data:
    data_path = './artificial_data'
else:
    data_path = './real_data'

Find all stroke team names:

In [3]:
make_teams = False

if make_teams:
    df_ml = pd.read_csv(f'{data_path}/ml_data.csv', low_memory=False)        
    stroke_teams = sorted(list(df_ml['stroke_team'].unique()))
    # Reshape for saving one unit per line:
    stroke_teams = np.array(stroke_teams).reshape(len(stroke_teams), 1)
    np.savetxt(os.path.join('output', 'stroke_teams.csv'), stroke_teams, fmt='%s')
else:
    stroke_teams = np.genfromtxt(os.path.join('output', 'stroke_teams.csv'), delimiter=',', dtype='U')

In [4]:
stroke_teams[:3]

array(["Addenbrooke's Hospital", 'Basildon University Hospital',
       'Blackpool Victoria Hospital'], dtype='<U51')

In [5]:
# Retrain models if required (otherwise load saved models)
train_models = False
replicates = 30

pm = IndividualPatientModel(
    data_path, train_models=train_models, replicates=replicates)

Define an example patient.

In [6]:
# Load prototype patient data
df_proto = pd.read_csv(f'{data_path}/prototype_patients.csv',
                       index_col='Patient prototype')

In [7]:
df_proto

,stroke_team,onset_to_arrival_time,onset_during_sleep,arrival_to_scan_time,infarction,stroke_severity,precise_onset_known,prior_disability,afib_anticoagulant,age
Patient prototype,,,,,,,,,,
Ideal,NaN,90,0,15,1,15,1,0,0,72.5
Late arrival,NaN,225,0,15,1,15,1,0,0,72.5
Mild,NaN,90,0,15,1,3,1,0,0,72.5
Prior disability,NaN,90,0,15,1,15,1,3,0,72.5
Imprecise,NaN,90,0,15,1,15,0,0,0,72.5
Age,NaN,90,0,15,1,15,1,0,0,87.5
Mild + Prior disability,NaN,90,0,15,1,3,1,3,0,72.5
Mild + Imprecise,NaN,90,0,15,1,3,0,0,0,72.5
Mild + Age,NaN,90,0,15,1,3,1,0,0,87.5


In [8]:
proto_names = df_proto.index.values

proto_names[:3]

array(['Ideal', 'Late arrival', 'Mild'], dtype=object)

Load data for onset-to-needle time:

In [9]:
df_hosp = pd.read_csv(f'{data_path}/data_for_sim.csv', index_col='stroke_team')

In [10]:
df_hosp.head()

,thrombolysis_rate,admissions,80_plus,onset_known,known_arrival_within_4hrs,onset_arrival_mins_mu,onset_arrival_mins_sigma,scan_within_4_hrs,arrival_scan_arrival_mins_mu,arrival_scan_arrival_mins_sigma,onset_scan_4_hrs,eligable,scan_needle_mins_mu,scan_needle_mins_sigma
stroke_team,,,,,,,,,,,,,,
Addenbrooke's Hospital,0.174868,2459,0.454386,0.647011,0.716530,4.657007,0.520571,0.952632,3.607126,0.691110,0.861878,0.428419,2.858108,1.013330
Basildon University Hospital,0.148209,1815,0.405440,0.690358,0.616121,4.634155,0.474288,0.963731,2.156707,1.537238,0.905914,0.382789,3.524274,0.694262
Blackpool Victoria Hospital,0.101740,1897,0.436709,0.463363,0.718999,4.515431,0.545704,0.955696,3.453097,0.773239,0.913907,0.311594,3.420102,0.830304
Bradford and Airedale SU,0.092749,2372,0.361955,0.453204,0.704186,4.573582,0.486773,0.953765,3.719591,0.834850,0.851801,0.334959,3.843458,0.598953
Bronglais Hospital,0.251309,382,0.463277,0.890052,0.520588,4.868520,0.418998,0.983051,2.116484,1.209905,0.902299,0.585987,3.621577,0.637036


In [11]:
df_hosp.columns

Index(['thrombolysis_rate', 'admissions', '80_plus', 'onset_known',
       'known_arrival_within_4hrs', 'onset_arrival_mins_mu',
       'onset_arrival_mins_sigma', 'scan_within_4_hrs',
       'arrival_scan_arrival_mins_mu', 'arrival_scan_arrival_mins_sigma',
       'onset_scan_4_hrs', 'eligable', 'scan_needle_mins_mu',
       'scan_needle_mins_sigma'],
      dtype='object')

## Run the outcomes

In [35]:
all_dist_dfs = []
ave_dist_dfs = []

cols_mrs = [f'mrs{i}' for i in range(7)]
# cols_untreated = [f'untreated_{m}' for m in cols_mrs]
# cols_treated = [f'treated_{m}' for m in cols_mrs]

dist_keys = ['less_3', 'more_4', 'weighted_mrs',
             'independent', 'dependent', 'dead']
stat_keys = ['', '_std', '_ci']
dist_keys_to_store = [f'treated_{d}{s if len(s) > 0 else ""}'
                      for d in dist_keys for s in stat_keys]
dist_keys_cols = [c.replace('treated_', '') for c in dist_keys_to_store]

n_combos = len(proto_names) * len(stroke_teams)
i_combo = 1
for proto_name in proto_names[:2]:
    # Pick out the patient data for this prototype:
    dict_patient = df_proto.loc[proto_name].to_dict()
    # Hard-code afib because missing from proto patient data:
    dict_patient['any_afib_diagnosis'] = dict_patient['afib_anticoagulant']
    
    for t, team in enumerate(stroke_teams[:2]):
        print(f'{100.0 * i_combo / n_combos:5.1f}%: {proto_name}, {team}'
              + ' '*40, end='\r')
        # Calculate onset-to-needle time for this patient:
        scan_to_needle = df_hosp.loc[team, 'scan_needle_mins_mu']
        onset_to_needle = (dict_patient['onset_to_arrival_time'] +
                           dict_patient['arrival_to_scan_time'] +
                           scan_to_needle)
        # Update the patient for this stroke team:
        dict_patient['stroke_team'] = team
        dict_patient['onset_to_thrombolysis'] = onset_to_needle

        # Run the model:
        pm.predict_patient(dict_patient, make_plot=False)

        # Store the results.
        # Distributions from each of the replicates:
        all_results_treated = pd.DataFrame(
            pm.treated_dist_each_model, columns=cols_mrs).copy()
        if t == 0:
            # Only store one copy of the no-treatment mRS values
            # because they're the same for all stroke units.
            all_results_untreated = pd.DataFrame(
                pm.untreated_dist_each_model, columns=cols_mrs).copy()
            # Round results:
            all_results = np.round(all_results_untreated, 6)
            all_results['model_no'] = np.arange(len(all_results))
            all_results['proto'] = proto_name
            all_results['stroke_team'] = 'any_team'
            all_results['treated'] = 0
            all_dist_dfs.append(all_results.copy())
        # Round results:
        all_results = np.round(all_results_treated, 6)
        all_results['model_no'] = np.arange(len(all_results))
        all_results['proto'] = proto_name
        all_results['stroke_team'] = team
        all_results['treated'] = 1
        all_dist_dfs.append(all_results.copy())
        

        # Average distributions:
        if t == 0:
            ave_arr = np.array([proto_name, 'any_team', 0] + [
                np.round(getattr(pm, k.replace('treated', 'untreated')), 5)
                for k in dist_keys_to_store])
            ave_arr = ave_arr.reshape(1, len(ave_arr))
            ave_results = pd.DataFrame(
                ave_arr, columns=['proto', 'stroke_team', 'treated'] + dist_keys_cols)
            ave_dist_dfs.append(ave_results.copy())

        ave_arr = np.array([proto_name, team, 1] + [
            np.round(getattr(pm, k), 5) for k in dist_keys_to_store])
        ave_arr = ave_arr.reshape(1, len(ave_arr))
        ave_results = pd.DataFrame(
            ave_arr, columns=['proto', 'stroke_team', 'treated'] + dist_keys_cols)
        ave_dist_dfs.append(ave_results.copy())
    i_combo += 1

# Convert to DataFrame:
df_all_dists = pd.concat(all_dist_dfs, axis='rows', ignore_index=True)
df_ave_dists = pd.concat(ave_dist_dfs, axis='rows', ignore_index=True)

  0.1%: Late arrival, Basildon University Hospital                                        

In [36]:
df_all_dists.to_csv(os.path.join('output', 'outcome_results_proto_patients_all_dists.csv'))
df_ave_dists.to_csv(os.path.join('output', 'outcome_results_proto_patients_averages.csv'))

## View results

In [37]:
df_all_dists

,mrs0,mrs1,mrs2,mrs3,mrs4,mrs5,mrs6,model_no,proto,stroke_team,treated
0,0.077076,0.082592,0.120123,0.123151,0.227302,0.037379,0.332376,0,Ideal,any_team,0
1,0.065986,0.118335,0.146222,0.165494,0.165238,0.067702,0.271023,1,Ideal,any_team,0
2,0.046637,0.084618,0.129336,0.139469,0.168150,0.085479,0.346310,2,Ideal,any_team,0
3,0.054711,0.084932,0.110471,0.176361,0.206385,0.053741,0.313398,3,Ideal,any_team,0
4,0.063470,0.104494,0.166581,0.192436,0.192404,0.040941,0.239673,4,Ideal,any_team,0
...,...,...,...,...,...,...,...,...,...,...,...
175,0.106887,0.185730,0.210574,0.037904,0.239009,0.070064,0.149833,25,Late arrival,Basildon University Hospital,1
176,0.089406,0.141034,0.203283,0.099259,0.240429,0.086152,0.140438,26,Late arrival,Basildon University Hospital,1
177,0.068420,0.160947,0.193564,0.135268,0.187670,0.048248,0.205882,27,Late arrival,Basildon University Hospital,1
178,0.123737,0.135597,0.148254,0.171017,0.195667,0.070211,0.155516,28,Late arrival,Basildon University Hospital,1


In [38]:
df_ave_dists

,proto,stroke_team,treated,less_3,less_3_std,less_3_ci,more_4,more_4_std,more_4_ci,weighted_mrs,...,weighted_mrs_ci,independent,independent_std,independent_ci,dependent,dependent_std,dependent_ci,dead,dead_std,dead_ci
0,Ideal,any_team,0,0.31223,0.03938,0.0147,0.3084,0.06872,0.02566,3.55679,...,0.0892,0.31223,0.03938,0.0147,0.43247,0.05342,0.01995,0.25531,0.07654,0.02858
1,Ideal,Addenbrooke's Hospital,1,0.5195,0.06581,0.02457,0.16494,0.04956,0.01851,2.70928,...,0.08775,0.5195,0.06581,0.02457,0.34256,0.0552,0.02061,0.13794,0.05028,0.01877
2,Ideal,Basildon University Hospital,1,0.56278,0.04848,0.0181,0.16774,0.03303,0.01233,2.60979,...,0.06527,0.56278,0.04848,0.0181,0.30634,0.04503,0.01682,0.13087,0.02852,0.01065
3,Late arrival,any_team,0,0.31223,0.03938,0.0147,0.3084,0.06872,0.02566,3.55679,...,0.0892,0.31223,0.03938,0.0147,0.43247,0.05342,0.01995,0.25531,0.07654,0.02858
4,Late arrival,Addenbrooke's Hospital,1,0.38689,0.04617,0.01724,0.24818,0.05955,0.02224,3.24872,...,0.07908,0.38689,0.04617,0.01724,0.40256,0.05979,0.02233,0.21054,0.06549,0.02445
5,Late arrival,Basildon University Hospital,1,0.43995,0.04516,0.01686,0.23252,0.04147,0.01548,3.10035,...,0.05223,0.43995,0.04516,0.01686,0.38386,0.05684,0.02122,0.17619,0.03378,0.01261


## Pick out best models

In [39]:
model_ids = df_all_dists['model_no'].unique()

model_ids

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])

For each combination of patient prototype and stroke team, find the mean score of each mRS bin. Find the model that gives the closest results to the average.

In [45]:
cols_mrs = [c for c in df_all_dists.columns if 'mrs' in c]

cols_best_models = ['proto', 'stroke_team', 'treated',
                    *[f'model_{c}_sum_sqres' for c in model_ids],
                    'best_model', 'best_sum_sqres']
list_best_models = []

for proto_name in proto_names[:2]:
    for team in ['any_team'] + list(stroke_teams[:2]):
        row = [proto_name, team]
        mask = (
            (df_all_dists['proto'] == proto_name) &
            (df_all_dists['stroke_team'] == team)
        )
        df_here = df_all_dists.loc[mask].copy()
        treated = df_here['treated'].values[0]
        row.append(treated)

        mean_dist = df_here[cols_mrs].mean()
        df_diff = df_here[cols_mrs] - mean_dist
        # Calculate sum of square residuals:
        series_diff = (df_diff**2.0).sum(axis='columns')
        series_diff.name = 'sum_sqres'
        # Put the model identifiers back in:
        df_diff = pd.concat((series_diff, df_here['model_no']), axis='columns')
        # Pick out the best model for each combo.
        # If there are joint winners then arbitrarily pick the first.
        best_sum_sqres = df_diff['sum_sqres'].min()
        best_model = df_diff.loc[np.round(df_diff['sum_sqres'], 5) == np.round(best_sum_sqres, 5)]['model_no'].values[0]
        row += list(np.round(df_diff['sum_sqres'], 5).values)
        row.append(best_model)
        row.append(round(best_sum_sqres, 5))
        list_best_models.append(row)

In [46]:
df_best_models = pd.DataFrame(list_best_models, columns=cols_best_models)

In [47]:
df_best_models.T

,0,1,2,3,4,5
proto,Ideal,Ideal,Ideal,Late arrival,Late arrival,Late arrival
stroke_team,any_team,Addenbrooke's Hospital,Basildon University Hospital,any_team,Addenbrooke's Hospital,Basildon University Hospital
treated,0,1,1,0,1,1
model_0_sum_sqres,0.0081,0.00753,0.00285,0.0081,0.00464,0.00438
model_1_sum_sqres,0.005,0.00906,0.02043,0.005,0.00868,0.0102
model_2_sum_sqres,0.01404,0.0118,0.02741,0.01404,0.00165,0.00764
model_3_sum_sqres,0.00615,0.00367,0.00771,0.00615,0.00385,0.00199
model_4_sum_sqres,0.00397,0.02689,0.0086,0.00397,0.00643,0.00952
model_5_sum_sqres,0.00354,0.00484,0.01383,0.00354,0.01144,0.01919
model_6_sum_sqres,0.00272,0.00574,0.01407,0.00272,0.0028,0.00185


In [48]:
df_best_models['model_6_sum_sqres']

0    0.00272
1    0.00574
2    0.01407
3    0.00272
4    0.00280
5    0.00185
Name: model_6_sum_sqres, dtype: float32

Compare some best models with the average values:

In [59]:
dict_treated_str = {0: 'no treatment', 1: 'treated'}

for proto_name in proto_names[:2]:
    for team in ['any_team'] + list(stroke_teams[:2]):
        mask = (
            (df_best_models['proto'] == proto_name) &
            (df_best_models['stroke_team'] == team)
        )
        df_here = df_best_models.loc[mask].copy()
        best_model = df_here['best_model'].values[0]
        treated = df_here['treated'].values[0]

        mask = (
            (df_all_dists['proto'] == proto_name) &
            (df_all_dists['stroke_team'] == team)
        )
        df_here = df_all_dists.loc[mask].copy()

        df_best = df_here.loc[df_here['model_no'] == best_model][cols_mrs]
        df_ave = df_here[cols_mrs].mean()

        print(f'{proto_name}, {dict_treated_str[treated]}, {team}')
        df = pd.concat((df_best, pd.DataFrame(df_ave).transpose()),
                       axis='rows', ignore_index=True)
        df['label'] = ['Best single model', 'Average across models']
        df = df.set_index('label')
        display(np.round(df, 3))

Ideal, no treatment, any_team


,mrs0,mrs1,mrs2,mrs3,mrs4,mrs5,mrs6
label,,,,,,,
Best single model,0.046,0.111,0.135,0.168,0.244,0.052,0.243
Average across models,0.065,0.103,0.145,0.150,0.229,0.053,0.255


Ideal, treated, Addenbrooke's Hospital


,mrs0,mrs1,mrs2,mrs3,mrs4,mrs5,mrs6
label,,,,,,,
Best single model,0.080,0.169,0.241,0.174,0.163,0.020,0.153
Average across models,0.104,0.194,0.221,0.152,0.164,0.027,0.138


Ideal, treated, Basildon University Hospital


...

,mrs0,mrs1,mrs2,mrs3,mrs4,mrs5,mrs6
label,,,,,,,
Best single model,0.046,0.111,0.135,0.168,0.244,0.052,0.243
Average across models,0.065,0.103,0.145,0.150,0.229,0.053,0.255


Late arrival, treated, Addenbrooke's Hospital


,mrs0,mrs1,mrs2,mrs3,mrs4,mrs5,mrs6
label,,,,,,,
Best single model,0.075,0.137,0.161,0.146,0.222,0.043,0.216
Average across models,0.075,0.133,0.179,0.154,0.211,0.038,0.211


Late arrival, treated, Basildon University Hospital


,mrs0,mrs1,mrs2,mrs3,mrs4,mrs5,mrs6
label,,,,,,,
Best single model,0.071,0.17,0.193,0.117,0.204,0.072,0.172
Average across models,0.083,0.14,0.217,0.123,0.205,0.056,0.176


## Test - pick off just one model

In [ ]:
print(stop, here, please)

In [ ]:
import pickle
outcome_models = pickle.load(
    open('./pickled_models/replicate_outcome_models.pkl', 'rb'))

In [ ]:
len(outcome_models)

In [ ]:
model_best = outcome_models[6]  # e.g.

In [ ]:
pickle.dump(model_best, open('./pickled_models/outcome_model_single.pkl', 'wb'))